In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('opd_clean.csv')
df.head()

,PatientNumber,RegistrationDate,Gender,Age,QueuedTo,ConsultDescription
0,010569291,2024-06-01 00:04:00,Female,44 Yr(s),GENERAL OPD,General Outpatient Care
1,010575732,2024-06-01 00:04:00,Female,78 Yr(s),GENERAL OPD,General Outpatient Care
2,010575731,2024-06-01 00:10:00,Female,2 Yr(s),ADMISSION,Admission
3,010454966,2024-06-01 01:22:00,FEMALE,31 Yr(s),GENERAL OPD,General Outpatient Care
4,010573369,2024-06-01 01:36:00,Female,84 Yr(s),GENERAL OPD,General Outpatient Care


In [2]:
# ------------------------------------------------
# Follow-Up / Return-Risk Model
# ------------------------------------------------


# Copy required data
followup_df = df.copy()

# ------------------------------------------------
# Clean fields
# ------------------------------------------------

followup_df["RegistrationDate"] = pd.to_datetime(
    followup_df["RegistrationDate"],
    errors="coerce"
)

followup_df["Age"] = pd.to_numeric(
    followup_df["Age"],
    errors="coerce"
)

followup_df["Gender"] = (
    followup_df["Gender"]
    .astype("string")
    .str.strip()
    .fillna("Unknown")
)

followup_df["QueuedTo"] = (
    followup_df["QueuedTo"]
    .astype("string")
    .str.strip()
)

# Remove records without patient ID/date
followup_df = followup_df.dropna(
    subset=[
        "PatientNumber",
        "RegistrationDate"
    ]
)

# Sort chronologically
followup_df = followup_df.sort_values(
    ["PatientNumber", "RegistrationDate"]
).reset_index(drop=True)

display(
    followup_df[
        [
            "PatientNumber",
            "RegistrationDate",
            "Gender",
            "Age",
            "QueuedTo"
        ]
    ].head()
)

,PatientNumber,RegistrationDate,Gender,Age,QueuedTo
0,010462209,2025-12-01 11:14:00,FEMALE,NaN,MCH
1,010462209,2025-12-19 11:58:00,FEMALE,NaN,MCH
2,010462209,2025-12-29 15:30:00,FEMALE,NaN,MCH
3,010462209,2026-01-09 07:52:00,FEMALE,NaN,ADMISSION
4,010462209,2026-08-11 11:47:00,FEMALE,NaN,EYE CLINIC


In [3]:
# ------------------------------------------------
# Create return-within-90-days target
# ------------------------------------------------

followup_df["next_visit_date"] = (
    followup_df
    .groupby("PatientNumber")["RegistrationDate"]
    .shift(-1)
)

followup_df["days_to_next_visit"] = (
    followup_df["next_visit_date"]
    - followup_df["RegistrationDate"]
).dt.days

followup_df["return_within_90d"] = (
    followup_df["days_to_next_visit"]
    .between(1, 90)
    .astype(int)
)

# Last visit of a patient has no future visit
# and therefore cannot provide a reliable target.
followup_model_df = followup_df[
    followup_df["next_visit_date"].notna()
].copy()

print(
    "Total records:",
    len(followup_model_df)
)

print(
    "\nReturn within 90 days:"
)

print(
    followup_model_df[
        "return_within_90d"
    ].value_counts()
)

print(
    "\nReturn rate:"
)

print(
    followup_model_df[
        "return_within_90d"
    ].mean()
)

Total records: 207660

Return within 90 days:
return_within_90d
1    171780
0     35880
Name: count, dtype: int64

Return rate:
0.8272175671771165


In [4]:
# ------------------------------------------------
# Historical patient features
# ------------------------------------------------

# Number of previous visits before current visit
followup_model_df["previous_visits"] = (
    followup_model_df
    .groupby("PatientNumber")
    .cumcount()
)

# Historical visit frequency
followup_model_df["total_visits_so_far"] = (
    followup_model_df["previous_visits"] + 1
)

# Days since previous visit
followup_model_df["days_since_previous_visit"] = (
    followup_model_df
    .groupby("PatientNumber")["RegistrationDate"]
    .diff()
    .dt.days
)

# Calendar features
followup_model_df["day_of_week"] = (
    followup_model_df[
        "RegistrationDate"
    ].dt.dayofweek
)

followup_model_df["month"] = (
    followup_model_df[
        "RegistrationDate"
    ].dt.month
)

followup_model_df["is_weekend"] = (
    followup_model_df[
        "day_of_week"
    ] >= 5
).astype(int)

# ------------------------------------------------
# Select model features
# ------------------------------------------------

features = [
    "Age",
    "Gender",
    "QueuedTo",
    "previous_visits",
    "total_visits_so_far",
    "days_since_previous_visit",
    "day_of_week",
    "month",
    "is_weekend"
]

target = "return_within_90d"

model_data = followup_model_df[
    features + [target, "RegistrationDate"]
].copy()

display(model_data.head())

,Age,Gender,QueuedTo,previous_visits,total_visits_so_far,days_since_previous_visit,day_of_week,month,is_weekend,return_within_90d,RegistrationDate
0,NaN,FEMALE,MCH,0,1,NaN,0,12,0,1,2025-12-01 11:14:00
1,NaN,FEMALE,MCH,1,2,18.0,4,12,0,1,2025-12-19 11:58:00
2,NaN,FEMALE,MCH,2,3,10.0,0,12,0,1,2025-12-29 15:30:00
3,NaN,FEMALE,ADMISSION,3,4,10.0,4,1,0,0,2026-01-09 07:52:00
6,NaN,FEMALE,MCH,0,1,NaN,3,6,0,1,2024-06-27 09:16:00


In [5]:
# ------------------------------------------------
# Handle missing values
# ------------------------------------------------

model_data["Age"] = model_data["Age"].fillna(
    model_data["Age"].median()
)

model_data["days_since_previous_visit"] = (
    model_data[
        "days_since_previous_visit"
    ].fillna(0)
)

model_data["Gender"] = (
    model_data["Gender"]
    .fillna("Unknown")
)

model_data["QueuedTo"] = (
    model_data["QueuedTo"]
    .fillna("Unknown")
)

print(
    model_data[
        features + [target]
    ].isna().sum()
)

Age                          207660
Gender                            0
QueuedTo                          0
previous_visits                   0
total_visits_so_far               0
days_since_previous_visit         0
day_of_week                       0
month                             0
is_weekend                        0
return_within_90d                 0
dtype: int64


In [6]:
# ------------------------------------------------
# Chronological train/test split
# ------------------------------------------------

model_data = model_data.sort_values(
    "RegistrationDate"
).reset_index(drop=True)

cutoff = model_data[
    "RegistrationDate"
].quantile(0.80)

train_data = model_data[
    model_data["RegistrationDate"] < cutoff
].copy()

test_data = model_data[
    model_data["RegistrationDate"] >= cutoff
].copy()

print(
    "Training period:",
    train_data["RegistrationDate"].min(),
    "to",
    train_data["RegistrationDate"].max()
)

print(
    "Testing period:",
    test_data["RegistrationDate"].min(),
    "to",
    test_data["RegistrationDate"].max()
)

print(
    "\nTrain size:",
    len(train_data)
)

print(
    "Test size:",
    len(test_data)
)

Training period: 2024-06-01 00:04:00 to 2026-02-21 09:22:00
Testing period: 2026-02-21 09:23:00 to 2026-08-18 23:06:00

Train size: 166127
Test size: 41533


In [7]:
# ------------------------------------------------
# Train return-risk LightGBM model
# ------------------------------------------------

import lightgbm as lgb

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

X_train = train_data[
    features
].copy()

y_train = train_data[
    target
]

X_test = test_data[
    features
].copy()

y_test = test_data[
    target
]

categorical_features = [
    "Gender",
    "QueuedTo"
]

for column in categorical_features:

    categories = sorted(
        set(X_train[column].astype(str))
        | set(X_test[column].astype(str))
    )

    X_train[column] = pd.Categorical(
        X_train[column].astype(str),
        categories=categories
    )

    X_test[column] = pd.Categorical(
        X_test[column].astype(str),
        categories=categories
    )

return_model = lgb.LGBMClassifier(
    objective="binary",
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42
)

return_model.fit(
    X_train,
    y_train,
    categorical_feature=categorical_features
)

print("Model trained successfully.")

[LightGBM] [Info] Number of positive: 134842, number of negative: 31285
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.005665 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 712
[LightGBM] [Info] Number of data points in the train set: 166127, number of used features: 8
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.811680 -> initscore=1.460965
[LightGBM] [Info] Start training from score 1.460965
Model trained successfully.


In [8]:
# ------------------------------------------------
# Evaluate return-risk model
# ------------------------------------------------

y_probability = return_model.predict_proba(
    X_test
)[:, 1]

y_prediction = (
    y_probability >= 0.50
).astype(int)

print(
    "Accuracy:",
    accuracy_score(
        y_test,
        y_prediction
    )
)

print(
    "Precision:",
    precision_score(
        y_test,
        y_prediction,
        zero_division=0
    )
)

print(
    "Recall:",
    recall_score(
        y_test,
        y_prediction,
        zero_division=0
    )
)

print(
    "F1:",
    f1_score(
        y_test,
        y_prediction,
        zero_division=0
    )
)

print(
    "ROC-AUC:",
    roc_auc_score(
        y_test,
        y_probability
    )
)

print(
    "\nClassification Report:"
)

print(
    classification_report(
        y_test,
        y_prediction,
        zero_division=0
    )
)

print(
    "\nConfusion Matrix:"
)

print(
    confusion_matrix(
        y_test,
        y_prediction
    )
)

Accuracy: 0.88640358269328
Precision: 0.8962319838654139
Recall: 0.9864908766040392
F1: 0.939197896798804
ROC-AUC: 0.6935502781445202

Classification Report:
              precision    recall  f1-score   support

           0       0.43      0.08      0.14      4595
           1       0.90      0.99      0.94     36938

    accuracy                           0.89     41533
   macro avg       0.66      0.53      0.54     41533
weighted avg       0.84      0.89      0.85     41533


Confusion Matrix:
[[  376  4219]
 [  499 36439]]


In [9]:
# ------------------------------------------------
# Feature importance
# ------------------------------------------------

feature_importance = pd.DataFrame({
    "feature": features,
    "importance": return_model.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

display(feature_importance)

,feature,importance
5,days_since_previous_visit,2669
3,previous_visits,1960
2,QueuedTo,1412
7,month,1381
6,day_of_week,1219
1,Gender,359
0,Age,0
4,total_visits_so_far,0
8,is_weekend,0


In [10]:
# ------------------------------------------------
# Patient return-risk scores
# ------------------------------------------------

risk_results = test_data[
    [
        "RegistrationDate",
        "Age",
        "Gender",
        "QueuedTo",
        "previous_visits",
        "total_visits_so_far",
        "days_since_previous_visit"
    ]
].copy()

risk_results["return_probability"] = (
    y_probability
)

risk_results["risk_category"] = pd.cut(
    risk_results["return_probability"],
    bins=[
        -np.inf,
        0.30,
        0.70,
        np.inf
    ],
    labels=[
        "Low",
        "Moderate",
        "High"
    ]
)

display(
    risk_results.head(20)
)

,RegistrationDate,Age,Gender,QueuedTo,previous_visits,total_visits_so_far,days_since_previous_visit,return_probability,risk_category
166127,2026-02-21 09:23:00,NaN,Male,RENAL,33,34,3.0,0.993494,High
166128,2026-02-21 09:23:00,NaN,Male,RENAL,182,183,3.0,0.989285,High
166129,2026-02-21 09:24:00,NaN,Male,RENAL,191,192,3.0,0.993028,High
166130,2026-02-21 09:25:00,NaN,Female,RENAL,189,190,3.0,0.993106,High
166131,2026-02-21 09:26:00,NaN,MALE,RENAL,123,124,3.0,0.995399,High
166132,2026-02-21 09:27:00,NaN,MALE,RENAL,75,76,3.0,0.998064,High
166133,2026-02-21 09:28:00,NaN,FEMALE,RENAL,168,169,2.0,0.970455,High
166134,2026-02-21 09:31:00,NaN,Female,RENAL,189,190,3.0,0.993106,High
166135,2026-02-21 09:31:00,NaN,Male,RENAL,10,11,4.0,0.981109,High
166136,2026-02-21 09:32:00,NaN,Male,RENAL,32,33,2.0,0.987987,High


In [11]:
df["QueuedTo"].value_counts()

QueuedTo
GENERAL OPD                            85368
MCH                                    29085
ADMISSION                              26041
CHRONIC CARE CLINIC (DM/HTN/TB/CCC)    25339
SPECIALITY CLINIC                      22312
ONCOLOGY                               13602
MEB SPECIALITY CLINIC                  13492
RENAL                                  11075
PAEDS BKKH                              9677
DENTAL                                  9249
OHNS                                    7273
EYE CLINIC                              6020
PHYSIOTHERAPY                           5821
DAYCASE                                 5444
Gen Surg OPD                            5137
CASUALTY                                3440
PRIVATE CLINIC                          3068
Audiology                               1378
PSYCHOLOGY                              1221
NEUROSURGERY                            1025
OCCUPATIONAL THERAPY                     814
NUTRITION                                747
T

RENAL ERROR:
NameError
name 'run_department_forecast_simulation' is not defined


NameError: name 'daily_7day' is not defined